In [ ]:
import numpy as np
import pandas as pd

In [ ]:
zero_day_attack = ["backdoor","mitm","ransomware"]

In [ ]:
file_path = r"data\NF_ToN_IoT.csv"

In [ ]:
chunks = pd.read_csv(
    file_path,
    chunksize=200000,
    low_memory=False
)

train_chunks = []
zero_day_chunks = []

for chunk in chunks:

    chunk["Attack"] = chunk["Attack"].str.lower()

 
    zd = chunk[
        chunk["Attack"].isin(zero_day_attack)
    ]

    normal = chunk[
        ~chunk["Attack"].isin(zero_day_attack)
    ]

    zero_day_chunks.append(zd)


    train_chunks.append(
        normal.sample(frac=0.02, random_state=42)
    )

train_data = pd.concat(train_chunks, ignore_index=True)
zero_day_test = pd.concat(zero_day_chunks, ignore_index=True)

print("Train shape:", train_data.shape)
print("Zero-day shape:", zero_day_test.shape)

In [ ]:
zero_day_test["Attack"].value_counts()

In [ ]:
train_data["Label"].value_counts()

In [ ]:
train_data["Attack"].value_counts()

In [ ]:
train_data.memory_usage(deep=True).sum() / 1024**2

In [ ]:
def reduce_memory(df):

    for col in df.columns:

        if df[col].dtype == "float64":
            df[col] = df[col].astype("float32")

        elif df[col].dtype == "int64":
            df[col] = df[col].astype("int32")

    return df


train_data = reduce_memory(train_data)
zero_day_test = reduce_memory(zero_day_test)

In [ ]:
train_data.memory_usage(deep=True).sum() / 1024**2

In [ ]:
train_data.columns

In [ ]:
remove_features = [
    "DNS_QUERY_ID",
    "DNS_QUERY_TYPE",
    "DNS_TTL_ANSWER",
    "FTP_COMMAND_RET_CODE",
    "ICMP_IPV4_TYPE"
]

train_data.drop(columns=remove_features,
                inplace=True,
                errors="ignore")

zero_day_test.drop(columns=remove_features,
                   inplace=True,
                   errors="ignore")

In [ ]:
train_data.shape, zero_day_test.shape

In [ ]:
numeric_cols = train_data.select_dtypes(
    include=[np.number]
).columns

In [ ]:
print(
    np.isinf(train_data[numeric_cols]).sum().sum()
)

In [ ]:
train_data[numeric_cols] = train_data[
    numeric_cols
].replace([np.inf, -np.inf], np.nan)

zero_day_test[numeric_cols] = zero_day_test[
    numeric_cols
].replace([np.inf, -np.inf], np.nan)

In [ ]:
median_vals = train_data[numeric_cols].median()

train_data[numeric_cols] = train_data[
    numeric_cols
].fillna(median_vals)

zero_day_test[numeric_cols] = zero_day_test[
    numeric_cols
].fillna(median_vals)

In [ ]:
print(
    np.isinf(train_data[numeric_cols]).sum().sum()
)

print(
    train_data[numeric_cols].isna().sum().sum()
)

In [ ]:
skewed_features = train_data.drop(
    ["Label","Attack"], axis=1
).skew()

log_cols = skewed_features[
    skewed_features > 1
].index

print("Log transforming:", list(log_cols))

In [ ]:
for col in log_cols:
    train_data[col] = np.log1p(train_data[col])
    zero_day_test[col] = np.log1p(zero_day_test[col])

In [ ]:
X_train = train_data.drop(
    columns=["Label", "Attack"]
)

y_train = train_data["Label"]

X_test = zero_day_test.drop(
    columns=["Label", "Attack"]
)

y_test = zero_day_test["Label"]

print(X_train.shape, X_test.shape)

In [ ]:
constant_cols = X_train.columns[
    X_train.nunique() <= 1
]

X_train.drop(columns=constant_cols, inplace=True)
X_test.drop(columns=constant_cols, inplace=True)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train.max()

In [ ]:
X_test.max()

In [ ]:
X_train.isna().sum().sum(), X_test.isna().sum().sum()

In [ ]:
X_train.isna().sum()[X_train.isna().sum() > 0]

In [ ]:
X_test.isna().sum()[X_test.isna().sum() > 0]

In [ ]:
median_vals = X_train.median()

In [ ]:
X_train = X_train.fillna(median_vals)
X_test = X_test.fillna(median_vals)

In [ ]:
X_train.isna().sum().sum(), X_test.isna().sum().sum()

In [ ]:
from sklearn.utils import resample

In [ ]:
Xy_train = pd.concat(
    [X_train, y_train],
    axis=1
)

majority = Xy_train[Xy_train["Label"] == 1]
minority = Xy_train[Xy_train["Label"] == 0]

majority_small = resample(
    majority,
    replace=False,
    n_samples=60000,
    random_state=42
)

train_small = pd.concat(
    [majority_small, minority]
)

In [ ]:
X_small = train_small.drop("Label", axis=1)
y_small = train_small["Label"]

In [ ]:
X_train_scaled = scaler.fit_transform(X_small)

X_test_scaled = scaler.transform(X_test)

In [ ]:
X_small.shape

In [ ]:
y_small.value_counts()

In [ ]:
np.isnan(X_train_scaled).sum(), np.isnan(X_test_scaled).sum()

In [ ]:
from imblearn.over_sampling import KMeansSMOTE

In [ ]:
from imblearn.over_sampling import KMeansSMOTE

smote = KMeansSMOTE(
    sampling_strategy="auto",   # VERY IMPORTANT
    k_neighbors=3,
    random_state=42,
    cluster_balance_threshold=0.1
)

X_train_bal, y_train_bal = smote.fit_resample(
    X_train_scaled,
    y_small
)

In [ ]:
y_train_bal.value_counts()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Reshape

In [ ]:
def build_q_network(state_size, action_size):

    model = Sequential()

    model.add(Dense(64, activation='relu',
                    input_shape=(state_size,)))
    model.add(Dropout(0.1))

    model.add(Reshape((1, 64)))

    model.add(LSTM(64, return_sequences=True))
    model.add(LSTM(64, return_sequences=True))
    model.add(LSTM(64))

    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.1))

    model.add(Dense(action_size, activation='linear'))

    model.compile(
        optimizer='adam',
        loss='mse'
    )

    return model

In [ ]:
class NIDSEnvironment:

    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.index = 0
        self.size = len(X)

    def reset(self):
        self.index = 0
        return self.X[self.index]

    def step(self, action):

        true_label = self.y[self.index]

        reward = 1 if action == true_label else -1

        self.index += 1
        done = self.index >= self.size

        next_state = None if done else self.X[self.index]

        return next_state, reward, done

In [ ]:
import random
from collections import deque
import numpy as np

class DQNAgent:

    def __init__(self, state_size, action_size):

        self.state_size = state_size
        self.action_size = action_size

        self.gamma = 0.97
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.996

        self.memory = deque(maxlen=2000)

        self.model = build_q_network(state_size, action_size)

    def act(self, state):

        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)

        q_values = self.model.predict(
            state.reshape(1, -1),
            verbose=0
        )

        return np.argmax(q_values[0])

    def remember(self, state, action, reward, next_state, done):
        self.memory.append(
            (state, action, reward, next_state, done)
        )

    def replay(self, batch_size):

        minibatch = random.sample(self.memory, batch_size)

        for state, action, reward, next_state, done in minibatch:

            target = reward

            if not done:
                next_q = self.model.predict(
                    next_state.reshape(1, -1),
                    verbose=0
                )[0]
                target += self.gamma * np.max(next_q)

            target_f = self.model.predict(
                state.reshape(1, -1),
                verbose=0
            )

            target_f[0][action] = target

            self.model.fit(
                state.reshape(1, -1),
                target_f,
                epochs=1,
                verbose=0
            )

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

In [ ]:
env = NIDSEnvironment(X_train_bal, y_train_bal)

state_size = X_train_bal.shape[1]
action_size = 2

agent = DQNAgent(state_size, action_size)

episodes = 5
batch_size = 64

for e in range(episodes):

    state = env.reset()
    done = False

    while not done:

        action = agent.act(state)

        next_state, reward, done = env.step(action)

        agent.remember(state, action, reward, next_state, done)

        state = next_state

        if len(agent.memory) > batch_size:
            agent.replay(batch_size)

    print("Episode:", e, "Epsilon:", agent.epsilon)

In [ ]:
env = NIDSEnvironment(X_train_bal, y_train_bal)

state_size = X_train_bal.shape[1]
action_size = 2

agent = DQNAgent(state_size, action_size)

episodes = 5
batch_size = 64

for e in range(episodes):

    state = env.reset()
    done = False

    while not done:

        action = agent.act(state)

        next_state, reward, done = env.step(action)

        agent.remember(state, action, reward, next_state, done)

        state = next_state

        if len(agent.memory) > batch_size:
            agent.replay(batch_size)

    print("Episode:", e, "Epsilon:", agent.epsilon)